In [ ]:
# =============================================================================
# jobs/23_ea_rt_live_readings.py
# The live tail of the EA real-time readings, for every site.
#
# WHY. The daily archive (job 10) is generated at 22:00 the day AFTER, and lands in
# bronze after that, so bronze is one to two days behind. A national monitoring map
# needs the last two days too. This job polls the EA API for the latest reading of
# every measure and keeps the last few days in ea_rt_readings_live. When a day's
# archive reaches bronze, job 10 releases that day from here: the archive is the
# authoritative record (it includes late telemetry), and this table is only the gap.
#
# What this script does:
#   1. Skips itself if the table was filled less than min_age_minutes ago, so it can
#      sit in more than one schedule (the polling master and its own 15-minute job)
#      without calling the API twice in a row.
#   2. Fetches the latest reading for every measure from the EA real-time API,
#      paging until a short page. A response with fewer measures than
#      EA_RT_LIVE_MIN_MEASURES is a bad response and fails the job.
#   3. Inserts readings not already held (match on measure and reading time), so a
#      re-run or an overlap never duplicates.
#   4. Deletes readings older than EA_RT_LIVE_KEEP_DAYS, a safety net for when the
#      archive hand-over has not run.
#   5. Refreshes two views: ea_rt_latest (the newest reading per measure) and
#      ea_rt_readings_all (the archive plus the live tail as one series, with the
#      archive winning for any day it holds).
#
# NO SPATIAL WORK. Station details (name, river, position, type) are in the register
# (09, 15). Join to it on measure_uri.
#
# Attribution (OGL), to be shown wherever these readings are displayed:
#   "this uses Environment Agency flood and river level data from the
#    real-time data API (Beta)"
#
# Exit values: "success" (readings written, possibly none new), "skipped_recent".
# A bad response raises, so a scheduled run is marked failed and alerts fire.
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import requests
from datetime import datetime, timezone
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType, DateType
)
from config import (
    TBL_EA_RT_LIVE, TBL_EA_RT_LATEST, TBL_EA_RT_ALL, TBL_EA_RT_BRONZE,
    EA_RT_LATEST_URL, EA_RT_LIVE_PAGE_LIMIT, EA_RT_LIVE_MIN_MEASURES, EA_RT_LIVE_KEEP_DAYS
)
from utils.helpers import get_spark, utc_now, table_exists, reading_rows_from_items

spark = get_spark()
now   = utc_now()

dbutils.widgets.text("min_age_minutes", "10")
min_age_minutes = float(dbutils.widgets.get("min_age_minutes") or 0)

LIVE_SCHEMA = StructType([
    StructField("measure_uri",  StringType(),    False),
    StructField("date_time",    TimestampType(), False),
    StructField("reading_date", DateType(),      False),
    StructField("value",        DoubleType(),    True),
    StructField("ingested_at",  TimestampType(), False),
])

# Created empty on first run, so everything that reads it works from the start.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TBL_EA_RT_LIVE} (
        measure_uri  STRING    NOT NULL,
        date_time    TIMESTAMP NOT NULL,
        reading_date DATE      NOT NULL,
        value        DOUBLE,
        ingested_at  TIMESTAMP NOT NULL
    ) USING DELTA
""")

if min_age_minutes > 0:
    # Worked out in SQL so the session time zone cannot skew it.
    age = spark.sql(f"""
        SELECT (unix_timestamp(current_timestamp()) - unix_timestamp(MAX(ingested_at))) / 60 AS age_min
        FROM {TBL_EA_RT_LIVE}
    """).collect()[0]["age_min"]
    if age is not None:
        if age < min_age_minutes:
            print(f"Live readings were filled {age:.1f} minutes ago (under {min_age_minutes:g}). Skipping.")
            dbutils.notebook.exit("skipped_recent")


In [ ]:
# =============================================================================
# FETCH THE LATEST READING OF EVERY MEASURE
# One call per page. The API returns the newest reading for each measure; a short
# page ends the loop. A timeout or error raises, which fails the run.
# =============================================================================

items  = []
offset = 0
while True:
    response = requests.get(
        EA_RT_LATEST_URL,
        params={"_limit": EA_RT_LIVE_PAGE_LIMIT, "_offset": offset},
        timeout=120,
    )
    response.raise_for_status()
    page = response.json().get("items", [])
    items.extend(page)
    print(f"  offset={offset}, got {len(page)} readings (total so far: {len(items)})")
    if len(page) < EA_RT_LIVE_PAGE_LIMIT:
        break
    offset += EA_RT_LIVE_PAGE_LIMIT

rows = reading_rows_from_items(items, now)
n_measures = len({r["measure_uri"] for r in rows})
print(f"Readings usable: {len(rows)}   measures: {n_measures}   (of {len(items)} items returned)")

if n_measures < EA_RT_LIVE_MIN_MEASURES:
    raise Exception(
        f"EA latest readings returned only {n_measures} measures (expected at least "
        f"{EA_RT_LIVE_MIN_MEASURES}). Treating it as a bad response; nothing was written."
    )


In [ ]:
# =============================================================================
# WRITE: INSERT NEW READINGS, THEN TRIM
# MERGE on (measure, time) inserts only readings not already held, so the same call
# twice, or two schedules overlapping, never duplicates. Nothing already stored is
# changed: a reading, once seen, stays as first seen until the archive replaces it.
# =============================================================================

incoming = [tuple(r[f.name] for f in LIVE_SCHEMA.fields) for r in rows]
spark.createDataFrame(incoming, LIVE_SCHEMA).createOrReplaceTempView("incoming_live_readings")

result = spark.sql(f"""
    MERGE INTO {TBL_EA_RT_LIVE} t
    USING incoming_live_readings s
      ON t.measure_uri = s.measure_uri AND t.date_time = s.date_time
    WHEN NOT MATCHED THEN INSERT *
""").collect()
inserted = int(result[0]["num_inserted_rows"]) if result and "num_inserted_rows" in result[0].asDict() else None
print(f"New readings inserted: {inserted if inserted is not None else 'n/a'} of {len(rows)} fetched.")

# Safety net: anything older than the keep window goes, whether or not job 10 has
# released it. The archive is the record for those days.
spark.sql(f"""
    DELETE FROM {TBL_EA_RT_LIVE}
    WHERE date_time < current_timestamp() - INTERVAL {int(EA_RT_LIVE_KEEP_DAYS)} DAYS
""")

spark.sql(f"""
    COMMENT ON TABLE {TBL_EA_RT_LIVE} IS
    'The live tail of EA real-time readings, every measure, polled every 15 to 30 minutes
     by 23_ea_rt_live_readings. One row per measure and reading time. It covers the one
     to two days before the daily archive (job 10) lands in ea_rt_readings_bronze; job 10
     releases each day from here once that day is archived, and anything older than
     EA_RT_LIVE_KEEP_DAYS is trimmed regardless. Read it through ea_rt_latest (newest
     per measure) or ea_rt_readings_all (archive plus this tail), not on its own.'
""")


In [ ]:
# =============================================================================
# THE TWO VIEWS
# ea_rt_latest      newest live reading per measure. The map colours from this.
# ea_rt_readings_all the archive plus the live tail as one series. Bronze wins: a live
#                   row is used only for a day bronze does not hold, so there is never
#                   a duplicate and a late archive correction always replaces the live
#                   value. source says which. Join to the register on measure_uri.
# =============================================================================

spark.sql(f"""
    CREATE OR REPLACE VIEW {TBL_EA_RT_LATEST} AS
    SELECT measure_uri, date_time, value, ingested_at
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY measure_uri ORDER BY date_time DESC) AS rn
        FROM {TBL_EA_RT_LIVE}
    )
    WHERE rn = 1
""")

if table_exists(spark, TBL_EA_RT_BRONZE):
    spark.sql(f"""
        CREATE OR REPLACE VIEW {TBL_EA_RT_ALL} AS
        SELECT measure_uri, date_time, reading_date, value, 'archive' AS source
        FROM {TBL_EA_RT_BRONZE}
        UNION ALL
        SELECT l.measure_uri, l.date_time, l.reading_date, l.value, 'live' AS source
        FROM {TBL_EA_RT_LIVE} l
        WHERE NOT EXISTS (SELECT 1 FROM {TBL_EA_RT_BRONZE} b WHERE b.reading_date = l.reading_date)
    """)
else:
    print(f"{TBL_EA_RT_BRONZE} does not exist yet (run 08), so {TBL_EA_RT_ALL} was not created.")

print(f"Live readings complete at {now.isoformat()}.")
dbutils.notebook.exit("success")
